# Notebook 5 — Checkpoint Loading & Resumed Training

Demonstrates loading a saved checkpoint and continuing training from that point.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/RL_Models'
except ImportError:
    SAVE_DIR = 'models'

import os, glob
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CheckpointCallback

gym.register_envs(ale_py)

In [ ]:
# Point to an existing checkpoint
RUN_DIR = os.path.join(SAVE_DIR, 'dqn_pong_default')

# Find the latest checkpoint automatically
ckpts = sorted(glob.glob(os.path.join(RUN_DIR, 'dqn_*_steps.zip')))
print('Available checkpoints:')
for c in ckpts:
    print(' ', c)

# Choose one to resume from (e.g. 500k)
RESUME_FROM = ckpts[4] if len(ckpts) > 4 else ckpts[-1]
print(f'\nResuming from: {RESUME_FROM}')

In [ ]:
ENV_ID = 'ALE/Pong-v5'
vec_env = make_atari_env(ENV_ID, n_envs=1, seed=42)
vec_env = VecFrameStack(vec_env, n_stack=4)

# Load the checkpoint — model weights AND optimizer state are restored
model = DQN.load(RESUME_FROM, env=vec_env)
print(f'Loaded model. Training resumes from timestep: {model.num_timesteps}')

In [ ]:
resume_dir = os.path.join(SAVE_DIR, 'dqn_pong_resumed')
checkpoint_cb = CheckpointCallback(
    save_freq=100_000,
    save_path=resume_dir,
    name_prefix='dqn_resumed',
)

# reset_num_timesteps=False preserves the timestep counter
model.learn(
    total_timesteps=500_000,
    callback=checkpoint_cb,
    reset_num_timesteps=False,
)

model.save(os.path.join(resume_dir, 'dqn_resumed_final'))
print(f'Resumed training complete. Final timestep: {model.num_timesteps}')